# 02. Дообучение HSEmotion и сравнение «до / после»

**До** — предобученная HSEmotion (AffectNet) без изменений. **После** — та же модель, дообученная
на кропах из `01_build_dataset`. Обе оцениваются на **одном и том же test**.

Правила, без которых сравнение нечестное:
1. Test — отдельные ролики, размечен вслепую (без подсказок модели).
2. Test не используется ни для обучения, ни для выбора эпохи или гиперпараметров (для этого есть val).
3. Главная метрика — **macro-F1** (классы несбалансированы, см. `docs/models.md`), дополнительно balanced accuracy.
4. Кадры одного ролика коррелированы, поэтому доверительные интервалы считаются бутстрэпом **по роликам**.

Результат прогона — папка `reports/<run_id>/`: `metrics.json`, предсказания, графики, веса `.pth` и `.onnx` для `ai-service`.

Обучение — на GPU (Colab/Kaggle T4 подойдёт). На CPU работает, но медленно: годится только для проверки ноутбука.

In [ ]:
# Colab / Kaggle:
# %pip install timm scikit-learn onnx onnxruntime
# from google.colab import drive; drive.mount("/content/drive")
# import os; os.environ["EMO_PROJECT_ROOT"] = "/content/drive/MyDrive/ai-live-emotions-decode"
#
# Локально (в отдельном окружении, не в .venv сервисов):
# %pip install torch torchvision timm scikit-learn pandas matplotlib scipy tqdm onnx onnxruntime pillow

## Конфигурация

In [ ]:
import copy
import json
import os
import random
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import v2 as T
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
_default_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = Path(os.environ.get("EMO_PROJECT_ROOT", _default_root)).resolve()
DATA_DIR = PROJECT_ROOT / "datasets" / "faces"
LABELS_CSV = DATA_DIR / "labels.csv"
WEIGHTS_DIR = PROJECT_ROOT / "datasets" / "weights"
REPORTS_DIR = PROJECT_ROOT / "reports"

CFG = dict(
    model_name="enet_b0_8_best_vgaf",  # на GPU попробуйте "enet_b2_8" (точнее, вход 260)
    epochs_head=3,          # этап 1: учим только последний слой
    epochs_full=15,         # этап 2: размораживаем всю сеть
    batch_size=32,
    lr_head=1e-3,
    lr_full=1e-4,
    weight_decay=1e-4,
    label_smoothing=0.1,
    balance="sampler",      # "sampler" | "loss" | None — как бороться с дисбалансом классов
    patience=5,             # ранняя остановка по macro-F1 на val
    include_dups=True,      # брать ли в train кропы, помеченные is_dup
    n_boot=2000,            # бутстрэп для доверительных интервалов
    bootstrap_by="clip",    # "clip" | "sample"
    seed=42,
    num_workers=2,
)
CLASSES: list[str] = json.loads((DATA_DIR / "classes.json").read_text())
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S") + "_" + CFG["model_name"]
RUN_DIR = REPORTS_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"


def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


seed_everything(CFG["seed"])
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print("run:", RUN_DIR)
print("classes:", CLASSES)

## Данные

In [ ]:
df = pd.read_csv(LABELS_CSV, dtype={"label": "string"})
df = df[df["label"].isin(CLASSES)].copy()
df["y"] = df["label"].map(CLASSES.index).astype(int)
# Дубли в val/test не берём: они завышают вес отдельных роликов.
df = df[(df["split"] == "train") & (CFG["include_dups"] | ~df["is_dup"]) | (df["split"] != "train") & ~df["is_dup"]]
splits = {s: df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"]}

counts = pd.crosstab(df["label"], df["split"]).reindex(index=CLASSES, columns=["train", "val", "test"], fill_value=0)
counts.loc["ВСЕГО"] = counts.sum()
counts.loc["роликов"] = [splits[s]["clip_id"].nunique() for s in counts.columns]
display(counts)

for s in ["train", "val", "test"]:
    assert len(splits[s]), f"сплит {s} пуст — разметьте его в 01_build_dataset"
leak = set(splits["train"]["clip_id"]) & set(splits["test"]["clip_id"])
assert not leak, f"ролики одновременно в train и test: {leak}"
missing_test = [c for c in CLASSES if counts.loc[c, "test"] == 0]
if missing_test:
    print("ВНИМАНИЕ: в test нет классов", missing_test, "— их F1 не измерить, macro-F1 считается по присутствующим")

In [ ]:
# Распределение классов по сплитам
BEFORE_C, AFTER_C = "#eb6834", "#2a78d6"   # «до» — оранжевый, «после» — синий (во всех графиках)
SPLIT_C = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}

share = counts.loc[CLASSES] / counts.loc[CLASSES].sum()
fig, ax = plt.subplots(figsize=(10, 3.5))
w = 0.27
x = np.arange(len(CLASSES))
for i, s in enumerate(["train", "val", "test"]):
    ax.bar(x + (i - 1) * w, share[s], w - 0.02, label=f"{s} (n={counts.loc['ВСЕГО', s]})", color=SPLIT_C[s])
ax.set_xticks(x, CLASSES)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Доля классов по сплитам")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(RUN_DIR / "class_distribution.png", dpi=120)
plt.show()

## Модель

In [ ]:
# --- Загрузка предобученной HSEmotion (EfficientNet, AffectNet) ---------------------------
# Порядок классов HSEmotion (idx_to_class из пакета hsemotion).
HSE_CLASSES_8 = ["anger", "contempt", "disgust", "fear", "happiness", "neutral", "sadness", "surprise"]
HSE_CLASSES_7 = ["anger", "disgust", "fear", "happiness", "neutral", "sadness", "surprise"]
HSE_MODELS = {
    # имя весов           : (архитектура timm, размер входа, классы)
    "enet_b0_8_best_vgaf": ("tf_efficientnet_b0", 224, HSE_CLASSES_8),
    "enet_b0_8_best_afew": ("tf_efficientnet_b0", 224, HSE_CLASSES_8),
    "enet_b0_8_va_mtl":    ("tf_efficientnet_b0", 224, HSE_CLASSES_8),  # +2 выхода valence/arousal
    "enet_b2_8":           ("tf_efficientnet_b2", 260, HSE_CLASSES_8),
    "enet_b2_7":           ("tf_efficientnet_b2", 260, HSE_CLASSES_7),
}
HSE_URL = "https://github.com/av-savchenko/face-emotion-recognition/raw/main/models/affectnet_emotions/{name}.pt"
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def load_hsemotion(name: str, weights_dir: Path) -> tuple["torch.nn.Module", int, list[str]]:
    """Возвращает (модель timm с весами HSEmotion, размер входа, классы в порядке выходов).

    Автор выложил веса как целиком сохранённую модель (pickle), поэтому первая загрузка
    идёт с weights_only=False — это исполняет код из файла, качайте только из репозитория автора.
    Дальше веса кэшируются как обычный state_dict и грузятся безопасно (weights_only=True).
    """
    import timm
    arch, size, classes = HSE_MODELS[name]
    weights_dir.mkdir(parents=True, exist_ok=True)
    sd_path = weights_dir / f"{name}.state_dict.pth"
    if not sd_path.exists():
        pt_path = weights_dir / f"{name}.pt"
        if not pt_path.exists():
            print("downloading", HSE_URL.format(name=name))
            torch.hub.download_url_to_file(HSE_URL.format(name=name), str(pt_path))
        legacy = torch.load(pt_path, map_location="cpu", weights_only=False)
        sd = {k.replace("classifier.0.", "classifier."): v for k, v in legacy.state_dict().items()}
        torch.save(sd, sd_path)
    sd = torch.load(sd_path, map_location="cpu", weights_only=True)
    model = timm.create_model(arch, pretrained=False, num_classes=sd["classifier.weight"].shape[0])
    model.load_state_dict(sd, strict=True)
    return model.eval(), size, classes


def restrict_head(model: "torch.nn.Module", src_classes: list[str], dst_classes: list[str]) -> "torch.nn.Module":
    """Оставляет в последнем слое только строки нужных классов (в порядке dst_classes).

    Так дообучение стартует ровно с предобученной модели, а у VA-моделей отбрасываются
    выходы valence/arousal.
    """
    missing = [c for c in dst_classes if c not in src_classes]
    assert not missing, f"классов {missing} нет в предобученной модели: {src_classes}"
    idx = torch.tensor([src_classes.index(c) for c in dst_classes])
    old = model.classifier
    new = torch.nn.Linear(old.in_features, len(dst_classes))
    with torch.no_grad():
        new.weight.copy_(old.weight[idx])
        new.bias.copy_(old.bias[idx])
    model.classifier = new
    return model

In [ ]:
def build_model() -> tuple[torch.nn.Module, int]:
    model, size, src_classes = load_hsemotion(CFG["model_name"], WEIGHTS_DIR)
    return restrict_head(model, src_classes, CLASSES), size


_, IMG_SIZE = build_model()

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.3, 0.3, 0.2, 0.02)], p=0.8),
    T.RandomRotation(10),
    T.RandomGrayscale(0.05),
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
# Как в HSEmotion при инференсе: просто resize до квадрата.
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class FaceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tf):
        self.paths = frame["crop_path"].tolist()
        self.y = frame["y"].tolist()
        self.tf = tf

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(Image.open(DATA_DIR / self.paths[i]).convert("RGB")), self.y[i]


def make_loader(split: str, train: bool) -> DataLoader:
    ds = FaceDataset(splits[split], train_tf if train else eval_tf)
    kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], pin_memory=device.type == "cuda")
    if train and CFG["balance"] == "sampler":
        freq = np.bincount(ds.y, minlength=len(CLASSES)).astype(float)
        weights = 1.0 / np.maximum(freq, 1)
        sampler = WeightedRandomSampler(weights[ds.y], num_samples=len(ds), replacement=True)
        return DataLoader(ds, sampler=sampler, drop_last=len(ds) > CFG["batch_size"], **kw)
    return DataLoader(ds, shuffle=train, drop_last=train and len(ds) > CFG["batch_size"], **kw)


loaders = {"train": make_loader("train", True), "val": make_loader("val", False), "test": make_loader("test", False)}
print({k: len(v.dataset) for k, v in loaders.items()}, "img_size:", IMG_SIZE)

## Метрики

In [ ]:
@torch.no_grad()
def predict(model: torch.nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval().to(device)
    out = []
    for x, _ in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(x.to(device, non_blocking=True)).float().softmax(1).cpu())
    return torch.cat(out).numpy()


def metrics(y: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    present = sorted(set(y))  # классы, которых нет в выборке, не тянут macro-F1 к нулю
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, labels=present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y, pred, labels=present, average="weighted", zero_division=0),
    }


def per_class_f1(y: np.ndarray, pred: np.ndarray) -> pd.Series:
    return pd.Series(f1_score(y, pred, labels=range(len(CLASSES)), average=None, zero_division=0), index=CLASSES)

## «До»: предобученная модель

In [ ]:
model_before, _ = build_model()
t0 = time.time()
prob_before = {s: predict(model_before, loaders[s]) for s in ["val", "test"]}
pred_before = {s: p.argmax(1) for s, p in prob_before.items()}
print(f"инференс: {time.time() - t0:.1f} c")
before = {s: metrics(splits[s]["y"].to_numpy(), pred_before[s]) for s in ["val", "test"]}
display(pd.DataFrame(before).T.round(4))

## Дообучение

Этап 1 — учим только последний слой (сеть заморожена, BatchNorm в eval). Этап 2 — дообучаем всю сеть с малым LR
и OneCycle. После каждой эпохи считаем macro-F1 на val, лучшую эпоху сохраняем.

In [ ]:
def class_weights() -> torch.Tensor | None:
    if CFG["balance"] != "loss":
        return None
    freq = np.bincount(splits["train"]["y"], minlength=len(CLASSES)).astype(float)
    w = freq.sum() / (len(CLASSES) * np.maximum(freq, 1))
    return torch.tensor(w, dtype=torch.float32, device=device)


def train_epoch(model, loader, optimizer, scheduler, scaler, criterion, freeze_backbone: bool) -> float:
    # При замороженной сети держим BatchNorm в eval, чтобы не портить его статистики маленьким датасетом.
    model.eval() if freeze_backbone else model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        total += loss.item() * len(y)
        n += len(y)
    return total / max(n, 1)


@torch.no_grad()
def val_loss(model, loader, criterion) -> float:
    model.eval()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            total += criterion(model(x), y).item() * len(y)
        n += len(y)
    return total / max(n, 1)


seed_everything(CFG["seed"])
model, _ = build_model()
model.to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights(), label_smoothing=CFG["label_smoothing"])
eval_criterion = torch.nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
y_val = splits["val"]["y"].to_numpy()

history = []
best = {"macro_f1": before["val"]["macro_f1"], "epoch": 0, "state": copy.deepcopy(model.state_dict())}
stale = 0
stages = [("head", CFG["epochs_head"]), ("full", CFG["epochs_full"])]
epoch = 0
for stage, n_epochs in stages:
    if n_epochs == 0:
        continue
    head_only = stage == "head"
    for p in model.parameters():
        p.requires_grad = not head_only
    for p in model.classifier.parameters():
        p.requires_grad = True
    params = [p for p in model.parameters() if p.requires_grad]
    lr = CFG["lr_head"] if head_only else CFG["lr_full"]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=CFG["weight_decay"])
    scheduler = None if head_only else torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=n_epochs * len(loaders["train"]), pct_start=0.1)
    stale = 0
    for _ in range(n_epochs):
        epoch += 1
        t0 = time.time()
        tr_loss = train_epoch(model, loaders["train"], optimizer, scheduler, scaler, criterion, head_only)
        v_pred = predict(model, loaders["val"]).argmax(1)
        m = metrics(y_val, v_pred)
        row = {"epoch": epoch, "stage": stage, "train_loss": tr_loss,
               "val_loss": val_loss(model, loaders["val"], eval_criterion), **{f"val_{k}": v for k, v in m.items()},
               "sec": time.time() - t0}
        history.append(row)
        improved = m["macro_f1"] > best["macro_f1"]
        if improved:
            best = {"macro_f1": m["macro_f1"], "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
            stale = 0
        else:
            stale += 1
        print(f"[{stage}] ep {epoch:2d} train_loss={tr_loss:.4f} val_loss={row['val_loss']:.4f} "
              f"val_macroF1={m['macro_f1']:.4f} bal_acc={m['balanced_accuracy']:.4f} {row['sec']:.0f}s {'*' if improved else ''}")
        if not head_only and stale >= CFG["patience"]:
            print("ранняя остановка")
            break

history = pd.DataFrame(history)
history.to_csv(RUN_DIR / "history.csv", index=False)
print(f"лучшая эпоха: {best['epoch']}, val macro-F1 = {best['macro_f1']:.4f}")
if best["epoch"] == 0:
    print("ВНИМАНИЕ: ни одна эпоха не обогнала исходную модель на val — «после» совпадёт с «до». "
          "Нужно больше данных или другие гиперпараметры.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
ax = axes[0]
ax.plot(history["epoch"], history["train_loss"], lw=2, color=AFTER_C, marker="o", ms=4, label="train")
ax.plot(history["epoch"], history["val_loss"], lw=2, color=BEFORE_C, marker="o", ms=4, label="val")
ax.set_title("Loss")
ax = axes[1]
ax.plot(history["epoch"], history["val_macro_f1"], lw=2, color=AFTER_C, marker="o", ms=4, label="дообучение")
ax.axhline(before["val"]["macro_f1"], color=BEFORE_C, lw=2, ls="--", label="«до»")
if best["epoch"]:
    ax.scatter([best["epoch"]], [best["macro_f1"]], s=80, color=AFTER_C, ec="white", lw=2, zorder=3)
ax.set_title("Macro-F1 на val")
for ax in axes:
    ax.set_xlabel("эпоха")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
    if CFG["epochs_head"] and len(history) > CFG["epochs_head"]:
        ax.axvline(CFG["epochs_head"] + 0.5, color="0.6", lw=1, ls=":")
plt.tight_layout()
plt.savefig(RUN_DIR / "learning_curves.png", dpi=120)
plt.show()

## «После»: оценка на test

In [ ]:
model.load_state_dict(best["state"])
torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "img_size": IMG_SIZE, "cfg": CFG,
            "arch": HSE_MODELS[CFG["model_name"]][0]}, RUN_DIR / "model.pth")

prob_after = predict(model, loaders["test"])
pred_after = prob_after.argmax(1)
y_test = splits["test"]["y"].to_numpy()
groups = splits["test"]["clip_id"].to_numpy()
after = metrics(y_test, pred_after)

## Сравнение «до / после»

- **Δ** — разница «после − до» с 95% доверительным интервалом (парный бутстрэп по роликам test: обе модели
  оцениваются на одних и тех же ресемплах). Если интервал Δ не содержит 0, улучшение устойчиво.
- **McNemar** — точный тест на кадрах, где ровно одна из моделей ошиблась. Он предполагает независимость кадров,
  поэтому при сильно коррелированных кадрах p-value занижен. Опирайтесь в первую очередь на интервал по роликам.

In [ ]:
def paired_bootstrap(y, p0, p1, groups, n_boot: int, by: str, seed: int) -> dict[str, dict[str, float]]:
    rng = np.random.default_rng(seed)
    if by == "clip":
        uniq = np.unique(groups)
        idx_by_group = [np.flatnonzero(groups == g) for g in uniq]
        if len(uniq) < 10:
            print(f"ВНИМАНИЕ: в test всего {len(uniq)} роликов — интервалы будут широкими и грубыми")
    keys = list(metrics(y, p0))
    samples = {k: {"before": [], "after": [], "delta": []} for k in keys}
    for _ in range(n_boot):
        if by == "clip":
            idx = np.concatenate([idx_by_group[i] for i in rng.integers(0, len(uniq), len(uniq))])
        else:
            idx = rng.integers(0, len(y), len(y))
        m0, m1 = metrics(y[idx], p0[idx]), metrics(y[idx], p1[idx])
        for k in keys:
            samples[k]["before"].append(m0[k]); samples[k]["after"].append(m1[k]); samples[k]["delta"].append(m1[k] - m0[k])
    return {k: {f"{part}_{q}": float(np.percentile(v, pct)) for part, v in d.items() for q, pct in (("lo", 2.5), ("hi", 97.5))}
            for k, d in samples.items()}


ci = paired_bootstrap(y_test, pred_before["test"], pred_after, groups, CFG["n_boot"], CFG["bootstrap_by"], CFG["seed"])

ok0, ok1 = pred_before["test"] == y_test, pred_after == y_test
fixed, broken = int((~ok0 & ok1).sum()), int((ok0 & ~ok1).sum())
mcnemar_p = float(binomtest(min(fixed, broken), fixed + broken, 0.5).pvalue) if fixed + broken else 1.0

rows = []
for k in before["test"]:
    b, a, c = before["test"][k], after[k], ci[k]
    rows.append({"метрика": k, "до": b, "после": a, "Δ": a - b,
                 "Δ 95% CI": f"[{c['delta_lo']:+.3f}, {c['delta_hi']:+.3f}]",
                 "до 95% CI": f"[{c['before_lo']:.3f}, {c['before_hi']:.3f}]",
                 "после 95% CI": f"[{c['after_lo']:.3f}, {c['after_hi']:.3f}]"})
summary = pd.DataFrame(rows).set_index("метрика")
display(summary.style.format({"до": "{:.4f}", "после": "{:.4f}", "Δ": "{:+.4f}"}))
print(f"test: {len(y_test)} кадров, {len(np.unique(groups))} роликов")
print(f"исправлено ошибок: {fixed}, новых ошибок: {broken}, McNemar p = {mcnemar_p:.4g}")

In [ ]:
# Метрики с интервалами: точка — значение на test, отрезок — 95% CI
keys = list(before["test"])
fig, ax = plt.subplots(figsize=(9, 3.2))
for j, (label, color, vals, part) in enumerate([("до", BEFORE_C, before["test"], "before"), ("после", AFTER_C, after, "after")]):
    ypos = np.arange(len(keys)) + (j - 0.5) * 0.3
    lo = [vals[k] - ci[k][f"{part}_lo"] for k in keys]
    hi = [ci[k][f"{part}_hi"] - vals[k] for k in keys]
    ax.errorbar([vals[k] for k in keys], ypos, xerr=[lo, hi], fmt="o", ms=8, color=color, ecolor=color,
                elinewidth=2, capsize=0, label=label, mec="white", mew=1.5)
    for k, yy in zip(keys, ypos):
        ax.annotate(f"{vals[k]:.3f}", (vals[k], yy), xytext=(0, 7), textcoords="offset points",
                    ha="center", fontsize=8, color="0.25")
ax.set_yticks(np.arange(len(keys)), keys)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_title(f"Test: до vs после (95% CI, бутстрэп по {'роликам' if CFG['bootstrap_by'] == 'clip' else 'кадрам'})")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(RUN_DIR / "metrics_ci.png", dpi=120)
plt.show()

In [ ]:
# Матрицы ошибок (нормированы по строкам: доля истинного класса)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (title, pred) in zip(axes, [("До", pred_before["test"]), ("После", pred_after)]):
    cm = confusion_matrix(y_test, pred, labels=range(len(CLASSES)))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            if cm[i, j]:
                ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=7,
                        color="white" if cmn[i, j] > 0.55 else "0.2")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("предсказание")
    ax.set_ylabel("истина")
    ax.set_title(f"{title}: macro-F1 = {metrics(y_test, pred)['macro_f1']:.3f}")
plt.tight_layout()
plt.savefig(RUN_DIR / "confusion_matrices.png", dpi=120)
plt.show()

In [ ]:
# F1 по классам
f1_b, f1_a = per_class_f1(y_test, pred_before["test"]), per_class_f1(y_test, pred_after)
support = pd.Series(np.bincount(y_test, minlength=len(CLASSES)), index=CLASSES)
fig, ax = plt.subplots(figsize=(9, 0.55 * len(CLASSES) + 1))
yy = np.arange(len(CLASSES))
ax.barh(yy - 0.2, f1_b, 0.36, color=BEFORE_C, label="до")
ax.barh(yy + 0.2, f1_a, 0.36, color=AFTER_C, label="после")
for i, c in enumerate(CLASSES):
    ax.text(max(f1_b[c], f1_a[c]) + 0.01, i, f"{f1_a[c] - f1_b[c]:+.2f}", va="center", fontsize=8, color="0.25")
ax.set_yticks(yy, [f"{c} (n={support[c]})" for c in CLASSES])
ax.invert_yaxis()
ax.set_xlim(0, 1.08)
ax.set_title("F1 по классам на test (подпись — Δ)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(RUN_DIR / "per_class_f1.png", dpi=120)
plt.show()

print("До:\n", classification_report(y_test, pred_before["test"], labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))
print("После:\n", classification_report(y_test, pred_after, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))

### Разбор ошибок: что исправилось и что сломалось

In [ ]:
def gallery(idx: np.ndarray, title: str, n: int = 16) -> None:
    if len(idx) == 0:
        print(title, "— нет примеров")
        return
    idx = np.random.default_rng(0).permutation(idx)[:n]
    cols = 8
    rows_n = (len(idx) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(2 * cols, 2.4 * rows_n), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        r = splits["test"].iloc[i]
        ax.imshow(Image.open(DATA_DIR / r.crop_path))
        ax.set_title(f"{r.label}\n{CLASSES[pred_before['test'][i]]} → {CLASSES[pred_after[i]]}", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


gallery(np.flatnonzero(~ok0 & ok1), "Исправлено (истина / до → после)")
gallery(np.flatnonzero(ok0 & ~ok1), "Сломалось (истина / до → после)")

## Сохранение отчёта

In [ ]:
report = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "cfg": CFG,
    "classes": CLASSES,
    "data": {s: {"n": len(splits[s]), "clips": int(splits[s]["clip_id"].nunique()),
                 "per_class": splits[s]["label"].value_counts().reindex(CLASSES, fill_value=0).to_dict()}
             for s in splits},
    "best_epoch": best["epoch"],
    "val": {"before": before["val"], "after_best_macro_f1": best["macro_f1"]},
    "test": {
        "before": before["test"], "after": after, "ci95": ci,
        "per_class_f1": {"before": f1_b.to_dict(), "after": f1_a.to_dict()},
        "mcnemar": {"fixed": fixed, "broken": broken, "p_value": mcnemar_p},
    },
}
(RUN_DIR / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=float))
(RUN_DIR / "config.json").write_text(json.dumps(CFG, ensure_ascii=False, indent=2))

pred_df = splits["test"][["crop_path", "clip_id", "label"]].copy()
pred_df["pred_before"] = [CLASSES[i] for i in pred_before["test"]]
pred_df["pred_after"] = [CLASSES[i] for i in pred_after]
for j, c in enumerate(CLASSES):
    pred_df[f"p_before_{c}"] = prob_before["test"][:, j].round(4)
    pred_df[f"p_after_{c}"] = prob_after[:, j].round(4)
pred_df.to_csv(RUN_DIR / "predictions_test.csv", index=False)
print("сохранено в", RUN_DIR)

## Экспорт в ONNX для `ai-service`

На входе — батч RGB-кропов `N×3×S×S` с нормализацией ImageNet (как `eval_tf`), на выходе — логиты по `CLASSES`.
Проверяем, что ONNX совпадает с PyTorch, и меряем задержку на CPU: бюджет из `docs/models.md` — 100 мс на кадр.

In [ ]:
import onnxruntime as ort

onnx_path = RUN_DIR / "model.onnx"
export_model = copy.deepcopy(model).cpu().eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(export_model, (dummy,), str(onnx_path), input_names=["input"], output_names=["logits"],
                  dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=64)}}, opset_version=18)
(RUN_DIR / "model.onnx.json").write_text(json.dumps(
    {"classes": CLASSES, "img_size": IMG_SIZE, "mean": IMAGENET_MEAN, "std": IMAGENET_STD, "input": "RGB"}, indent=2))

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
x = torch.stack([eval_tf(Image.open(DATA_DIR / p).convert("RGB")) for p in splits["test"]["crop_path"][:8]])
with torch.no_grad():
    ref = export_model(x).numpy()
got = sess.run(None, {"input": x.numpy()})[0]
print("max |onnx - torch| =", float(np.abs(ref - got).max()))

one = x[:1].numpy()
for _ in range(5):
    sess.run(None, {"input": one})
t = []
for _ in range(30):
    t0 = time.perf_counter(); sess.run(None, {"input": one}); t.append((time.perf_counter() - t0) * 1000)
print(f"CPU, 1 лицо: median {np.median(t):.1f} мс, p95 {np.percentile(t, 95):.1f} мс")
report["onnx_cpu_ms"] = {"median": float(np.median(t)), "p95": float(np.percentile(t, 95))}
_ = (RUN_DIR / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=float))

## Все прогоны

In [ ]:
runs = []
for f in sorted(REPORTS_DIR.glob("*/metrics.json")):
    r = json.loads(f.read_text())
    t = r["test"]
    runs.append({"run": r["run_id"], "model": r["cfg"]["model_name"], "train_n": r["data"]["train"]["n"],
                 "test_n": r["data"]["test"]["n"], "best_epoch": r["best_epoch"],
                 "macro_f1_before": t["before"]["macro_f1"], "macro_f1_after": t["after"]["macro_f1"],
                 "Δ macro_f1": t["after"]["macro_f1"] - t["before"]["macro_f1"],
                 "Δ CI": f"[{t['ci95']['macro_f1']['delta_lo']:+.3f}, {t['ci95']['macro_f1']['delta_hi']:+.3f}]",
                 "bal_acc_after": t["after"]["balanced_accuracy"]})
display(pd.DataFrame(runs).round(4))
print("Сравнивать прогоны между собой корректно, только если у них одинаковый test (те же ролики и метки).")